# Release16 Guarded Reward Action / Feature Diagnostic

This notebook diagnoses the failed `release16_guarded_reward_short_ppo_screen_v1` result before any further training.

- Latest result: `reward_ppo_selected` failed the PPO-vs-BC drift guard and only passed `1/3` primary 20% walk-forward gates.
- Comparison pair: latest `reward_ppo_selected` vs previous `bc_initialization` from the same short PPO screen.
- Baseline family: converted SOXLBacktest qlib reference and release16 bull-reentry BC warm-start lineage.
- Role: `diagnose`; no actor promotion or retention is allowed here.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Changed layer being diagnosed: `reward`, `bc_to_ppo_handoff`, `policy_action_distribution`.

The primary question is whether the failed reward PPO path lost performance because it took less trend exposure, took risk in the wrong state, or because current target/reward features do not separate the failed state.


In [1]:
from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

RUN_CWD = Path.cwd().resolve()
for candidate_root in (RUN_CWD, *RUN_CWD.parents):
    if (candidate_root / "SPEC.md").exists() and (candidate_root / "src" / "soxl_quant").exists():
        ROOT = candidate_root
        break
else:
    raise RuntimeError(f"could not locate repo root from {RUN_CWD}")
for module_path in (ROOT, ROOT / "src"):
    module_path_text = str(module_path)
    if module_path_text not in sys.path:
        sys.path.insert(0, module_path_text)

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.path_diagnostics import (
    compare_interval_feature_deltas,
    find_worst_drawdown_intervals,
    summarize_interval_feature_groups,
)
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
EXPERIMENT = "release16_guarded_reward_action_feature_diagnostic_v1"
CARD = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_CARD = ROOT / "runtime" / "reward" / "card" / "release16_guarded_reward_short_ppo_screen_v1"
SOURCE_SUMMARY = SOURCE_CARD / "summary.json"
SOURCE_FOLD_RESULTS = SOURCE_CARD / "reward_ppo_fold_results.csv"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
STRICT_MAX_DD = 15.0
ROLLING_WINDOW = 21

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

KEY_FEATURES = [
    "mom_21",
    "mom_63",
    "trend_gap_21",
    "trend_gap_63",
    "anchor_gap_180",
    "anchor_slope_180",
    "regime_risk",
    "realized_vol_21",
    "vol_of_vol_21",
    "drawdown_63",
    "vix_percentile_252",
    "relative_strength_21",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    if pd.isna(value):
        return None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))

if not SOURCE_SUMMARY.exists() or not SOURCE_FOLD_RESULTS.exists():
    raise FileNotFoundError("source short PPO screen outputs are missing")
print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/reward/card/release16_guarded_reward_action_feature_diagnostic_v1


In [3]:
source_summary = load_json(SOURCE_SUMMARY)
fold_results = pd.read_csv(SOURCE_FOLD_RESULTS)
profile = load_profile(PROFILE_NAME)

init_qlib(QLIB_DATA_PATH)
features_by_fold: dict[str, pd.DataFrame] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    test_df = segment_frame(dataset, "test").copy()
    test_df.index = pd.to_datetime(test_df.index).normalize()
    features_by_fold[fold] = test_df

print("source decision", source_summary.get("decision"))
print("fold rows", fold_results.shape)
fold_results[["fold", "bc_test_relative_return", "bc_test_max_dd", "selected_test_relative_return", "selected_test_max_dd", "ppo_vs_bc_relative_delta", "ppo_vs_bc_max_dd_delta"]]


[33158:MainThread](2026-06-15 05:13:28,498) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[33158:MainThread](2026-06-15 05:13:28,854) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[33158:MainThread](2026-06-15 05:13:28,855) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[33158:MainThread](2026-06-15 05:13:29,705) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[33158:MainThread](2026-06-15 05:13:29,705) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[33158:MainThread](2026-06-15 05:13:32,036) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[33158:MainThread](2026-06-15 05:13:32,037) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[33158:MainThread](2026-06-15 05:13:32,037) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


source decision release16_guarded_reward_short_ppo_drift_failed_no_promotion
fold rows (3, 67)


,fold,bc_test_relative_return,bc_test_max_dd,selected_test_relative_return,selected_test_max_dd,ppo_vs_bc_relative_delta,ppo_vs_bc_max_dd_delta
0,wf_2020,22.281639,21.106156,-15.260823,21.112771,-37.542462,0.006615
1,wf_2022,72.450860,15.072639,72.450860,15.072639,0.000000,0.000000
2,wf_2023_2024,14.302062,13.959434,14.302062,13.959434,0.000000,0.000000


In [4]:
def load_daily(path_text: str, *, label: str, fold: str) -> pd.DataFrame:
    path = ROOT / str(path_text)
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    frame["datetime"] = pd.to_datetime(frame["datetime"]).dt.normalize()
    frame = frame.sort_values("datetime").reset_index(drop=True)
    frame["fold"] = fold
    frame["label"] = label
    frame["account"] = pd.to_numeric(frame["strategy_nav"], errors="coerce")
    frame["nav"] = frame["account"] / float(frame["account"].iloc[0])
    frame["strategy_return_pct"] = frame["nav"].pct_change().fillna(0.0) * 100.0
    frame["buyhold_nav_rebased"] = pd.to_numeric(frame["buyhold_nav"], errors="coerce") / float(frame["buyhold_nav"].iloc[0])
    frame["buyhold_return_pct"] = frame["buyhold_nav_rebased"].pct_change().fillna(0.0) * 100.0
    return frame


def max_drawdown_pct(nav: pd.Series) -> float:
    values = pd.to_numeric(nav, errors="coerce").dropna()
    if values.empty:
        return 0.0
    peak = values.cummax().replace(0.0, np.nan)
    return float((1.0 - values / peak).fillna(0.0).max() * 100.0)


def path_return_pct(nav: pd.Series) -> float:
    values = pd.to_numeric(nav, errors="coerce").dropna()
    if len(values) < 2:
        return 0.0
    start = float(values.iloc[0])
    end = float(values.iloc[-1])
    return float((end / start - 1.0) * 100.0) if start else 0.0

all_daily: list[pd.DataFrame] = []
for row in fold_results.to_dict("records"):
    fold = str(row["fold"])
    all_daily.append(load_daily(row["bc_test_daily_path"], label="bc_initialization", fold=fold))
    all_daily.append(load_daily(row["selected_test_daily_path"], label="reward_ppo_selected", fold=fold))
daily = pd.concat(all_daily, ignore_index=True)
daily.to_csv(CARD / "daily_paths_merged.csv", index=False)

position_rows = []
for (fold, label), sub in daily.groupby(["fold", "label"], sort=True):
    actions = sub["action"].astype(str).value_counts().to_dict() if "action" in sub.columns else {}
    nav = pd.to_numeric(sub["nav"], errors="coerce")
    bh = pd.to_numeric(sub["buyhold_nav_rebased"], errors="coerce")
    position = pd.to_numeric(sub["position"], errors="coerce").fillna(0.0)
    position_rows.append(
        {
            "fold": fold,
            "candidate": label,
            "rows": int(len(sub)),
            "return_pct": path_return_pct(nav),
            "max_dd_pct": max_drawdown_pct(nav),
            "buyhold_return_pct": path_return_pct(bh),
            "relative_return_pp": path_return_pct(nav) - path_return_pct(bh),
            "avg_pos": float(position.mean()),
            "median_pos": float(position.median()),
            "pos_p10": float(position.quantile(0.10)),
            "pos_p90": float(position.quantile(0.90)),
            "min_pos": float(position.min()),
            "max_pos": float(position.max()),
            "buy_days": int(actions.get("BUY", 0)),
            "sell_days": int(actions.get("SELL", 0)),
            "hold_days": int(actions.get("HOLD", 0)),
            "flat_days": int((position <= 0.01).sum()),
            "high_exposure_days_65": int((position >= 0.65).sum()),
            "low_exposure_days_25": int((position <= 0.25).sum()),
        }
    )
position_summary = pd.DataFrame(position_rows)
position_summary.to_csv(CARD / "position_action_summary.csv", index=False)
display(position_summary)


,fold,candidate,rows,return_pct,max_dd_pct,buyhold_return_pct,relative_return_pp,avg_pos,median_pos,pos_p10,pos_p90,min_pos,max_pos,buy_days,sell_days,hold_days,flat_days,high_exposure_days_65,low_exposure_days_25
0,wf_2020,bc_initialization,252,80.375210,21.106156,58.093571,22.281639,0.425202,0.450023,0.228675,0.679370,0.103727,0.950000,14,18,220,0,35,38
1,wf_2020,reward_ppo_selected,252,42.832748,21.112771,58.093571,-15.260823,0.398541,0.450000,0.137523,0.664055,0.100000,0.950000,11,20,221,0,31,50
2,wf_2022,bc_initialization,250,-14.137212,15.072639,-86.588072,72.450860,0.087837,0.094310,0.000000,0.154981,0.000000,0.450000,14,11,225,56,0,234
3,wf_2022,reward_ppo_selected,250,-14.137212,15.072639,-86.588072,72.450860,0.087837,0.094310,0.000000,0.154981,0.000000,0.450000,14,11,225,56,0,234
4,wf_2023_2024,bc_initialization,251,13.945428,13.959434,-0.356635,14.302062,0.197766,0.161382,0.053752,0.453965,0.000000,0.660876,13,17,221,19,6,195
5,wf_2023_2024,reward_ppo_selected,251,13.945428,13.959434,-0.356635,14.302062,0.197766,0.161382,0.053752,0.453965,0.000000,0.660876,13,17,221,19,6,195


In [5]:
daily_delta_rows: list[pd.DataFrame] = []
rolling_rows: list[dict[str, Any]] = []

for fold in WALK_FORWARD_WINDOWS:
    bc = daily[(daily["fold"].eq(fold)) & daily["label"].eq("bc_initialization")].copy()
    ppo = daily[(daily["fold"].eq(fold)) & daily["label"].eq("reward_ppo_selected")].copy()
    merged = bc[["datetime", "strategy_nav", "buyhold_nav", "price", "position", "action"]].merge(
        ppo[["datetime", "strategy_nav", "position", "action"]],
        on="datetime",
        how="inner",
        suffixes=("_bc", "_ppo"),
    )
    merged["fold"] = fold
    merged["bc_nav"] = merged["strategy_nav_bc"] / float(merged["strategy_nav_bc"].iloc[0])
    merged["ppo_nav"] = merged["strategy_nav_ppo"] / float(merged["strategy_nav_ppo"].iloc[0])
    merged["buyhold_nav_rebased"] = merged["buyhold_nav"] / float(merged["buyhold_nav"].iloc[0])
    merged["ppo_minus_bc_nav_pp"] = (merged["ppo_nav"] - merged["bc_nav"]) * 100.0
    merged["position_delta"] = pd.to_numeric(merged["position_ppo"], errors="coerce") - pd.to_numeric(merged["position_bc"], errors="coerce")
    merged["action_changed"] = merged["action_bc"].astype(str) != merged["action_ppo"].astype(str)
    features = features_by_fold[fold]
    merged = merged.merge(features.reset_index().rename(columns={"index": "datetime"}), on="datetime", how="left")
    daily_delta_rows.append(merged)

    for idx in range(0, max(0, len(merged) - ROLLING_WINDOW + 1)):
        window = merged.iloc[idx : idx + ROLLING_WINDOW].copy()
        bc_ret = path_return_pct(window["bc_nav"])
        ppo_ret = path_return_pct(window["ppo_nav"])
        bh_ret = path_return_pct(window["buyhold_nav_rebased"])
        rolling_rows.append(
            {
                "fold": fold,
                "start": str(window["datetime"].iloc[0].date()),
                "end": str(window["datetime"].iloc[-1].date()),
                "rows": int(len(window)),
                "bc_return_pct": bc_ret,
                "ppo_return_pct": ppo_ret,
                "ppo_minus_bc_return_pp": float(ppo_ret - bc_ret),
                "buyhold_return_pct": bh_ret,
                "avg_bc_pos": float(window["position_bc"].mean()),
                "avg_ppo_pos": float(window["position_ppo"].mean()),
                "avg_position_delta": float(window["position_delta"].mean()),
                "action_changed_days": int(window["action_changed"].sum()),
                "max_bc_dd_pct": max_drawdown_pct(window["bc_nav"]),
                "max_ppo_dd_pct": max_drawdown_pct(window["ppo_nav"]),
            }
        )

daily_delta = pd.concat(daily_delta_rows, ignore_index=True)
rolling_loss_windows = pd.DataFrame(rolling_rows).sort_values(["ppo_minus_bc_return_pp", "fold", "start"]).reset_index(drop=True)
daily_delta.to_csv(CARD / "daily_position_deltas.csv", index=False)
rolling_loss_windows.to_csv(CARD / "rolling_loss_windows.csv", index=False)

display(rolling_loss_windows.head(12))


,fold,start,end,rows,bc_return_pct,ppo_return_pct,ppo_minus_bc_return_pp,buyhold_return_pct,avg_bc_pos,avg_ppo_pos,avg_position_delta,action_changed_days,max_bc_dd_pct,max_ppo_dd_pct
0,wf_2020,2020-11-05,2020-12-04,21,16.302958,1.945125,-14.357832,39.436081,0.450023,0.450445,0.000422,8,4.876725,8.984484
1,wf_2020,2020-11-09,2020-12-08,21,21.345524,7.227663,-14.117861,41.445134,0.450023,0.450445,0.000422,9,2.177847,2.563070
2,wf_2020,2020-11-06,2020-12-07,21,12.903692,-0.725653,-13.629345,34.348311,0.450023,0.450445,0.000422,8,4.876725,8.984484
3,wf_2020,2020-10-22,2020-11-19,21,15.732014,3.599630,-12.132384,24.088576,0.450023,0.450445,0.000422,9,10.612394,16.768701
4,wf_2020,2020-11-18,2020-12-17,21,14.906026,2.847646,-12.058381,31.000662,0.450023,0.450445,0.000422,8,10.848293,8.444598
5,wf_2020,2020-11-02,2020-12-01,21,35.096621,23.089201,-12.007420,69.003007,0.450023,0.450445,0.000422,8,4.876725,8.984484
6,wf_2020,2020-11-03,2020-12-02,21,30.514542,18.634523,-11.880019,60.725351,0.450023,0.450445,0.000422,7,4.876725,8.984484
7,wf_2020,2020-11-16,2020-12-15,21,13.120438,1.274058,-11.846381,24.707106,0.450023,0.450445,0.000422,8,10.848293,8.444598
8,wf_2020,2020-11-17,2020-12-16,21,13.685655,1.946727,-11.738928,27.538136,0.450023,0.450445,0.000422,9,10.848293,8.444598
9,wf_2020,2020-11-04,2020-12-03,21,22.105160,10.990244,-11.114916,45.697407,0.450023,0.450445,0.000422,7,4.876725,8.984484


In [6]:
interval_rows: list[pd.DataFrame] = []
feature_delta_frames: list[pd.DataFrame] = []
feature_group_frames: list[pd.DataFrame] = []

for fold in WALK_FORWARD_WINDOWS:
    fold_features = features_by_fold[fold].reset_index().rename(columns={"index": "datetime"})
    for label in ("bc_initialization", "reward_ppo_selected"):
        path = daily[(daily["fold"].eq(fold)) & daily["label"].eq(label)].copy()
        enriched = path.merge(fold_features, on="datetime", how="left")
        diag = enriched.copy()
        intervals = find_worst_drawdown_intervals(diag, top_n=3, min_depth=0.05)
        if not intervals.empty:
            intervals.insert(0, "candidate", label)
            intervals.insert(0, "fold", fold)
            interval_rows.append(intervals)

    ppo_path = daily[(daily["fold"].eq(fold)) & daily["label"].eq("reward_ppo_selected")].copy().merge(fold_features, on="datetime", how="left")
    bc_path = daily[(daily["fold"].eq(fold)) & daily["label"].eq("bc_initialization")].copy().merge(fold_features, on="datetime", how="left")
    ppo_diag = ppo_path.copy()
    bc_diag = bc_path.copy()

    # Feature deltas for the worst PPO-vs-BC rolling loss windows. This catches underparticipation loss even when MaxDD is unchanged.
    worst_windows = rolling_loss_windows[rolling_loss_windows["fold"].eq(fold)].head(3).copy()
    interval_like = worst_windows.rename(columns={"start": "peak_date", "end": "trough_date"})
    interval_like.insert(0, "rank", range(1, len(interval_like) + 1))
    deltas = compare_interval_feature_deltas(
        candidate_daily=ppo_diag,
        baseline_daily=bc_diag,
        intervals=interval_like,
        feature_cols=[c for c in KEY_FEATURES if c in ppo_diag.columns],
    )
    if not deltas.empty:
        deltas.insert(0, "interval_type", "worst_rolling_ppo_vs_bc_loss")
        deltas.insert(0, "fold", fold)
        feature_delta_frames.append(deltas)
        groups = summarize_interval_feature_groups(deltas, top_n=8)
        groups.insert(0, "interval_type", "worst_rolling_ppo_vs_bc_loss")
        groups.insert(0, "fold", fold)
        feature_group_frames.append(groups)

intervals_all = pd.concat(interval_rows, ignore_index=True) if interval_rows else pd.DataFrame()
feature_deltas = pd.concat(feature_delta_frames, ignore_index=True) if feature_delta_frames else pd.DataFrame()
feature_groups = pd.concat(feature_group_frames, ignore_index=True) if feature_group_frames else pd.DataFrame()
intervals_all.to_csv(CARD / "drawdown_intervals.csv", index=False)
feature_deltas.to_csv(CARD / "feature_deltas.csv", index=False)
feature_groups.to_csv(CARD / "feature_group_summary.csv", index=False)

display(intervals_all.head(20))
display(feature_groups)


,fold,candidate,rank,peak_date,trough_date,start,end,rows,max_dd,max_dd_fraction,return,avg_pos,peak_nav,trough_nav
0,wf_2020,bc_initialization,1,2020-09-02,2020-09-08,2020-09-02,2020-09-08,4,21.1062,0.211062,-21.1062,0.460278,1.503164,1.185904
1,wf_2020,bc_initialization,2,2020-12-08,2020-12-11,2020-12-08,2020-12-11,4,10.8483,0.108483,-10.8483,0.450023,1.907381,1.700462
2,wf_2020,bc_initialization,3,2020-06-10,2020-06-26,2020-06-10,2020-06-26,13,7.1555,0.071555,-7.1555,0.489456,1.217756,1.130619
3,wf_2020,reward_ppo_selected,1,2020-09-02,2020-09-08,2020-09-02,2020-09-08,4,21.1128,0.211128,-21.1128,0.106936,1.428666,1.127035
4,wf_2020,reward_ppo_selected,2,2020-11-06,2020-11-10,2020-11-06,2020-11-10,3,8.9845,0.089845,-8.9845,0.450445,1.476796,1.344114
5,wf_2020,reward_ppo_selected,3,2020-12-08,2020-12-11,2020-12-08,2020-12-11,4,8.4446,0.084446,-8.4446,0.450445,1.479174,1.354264
6,wf_2022,bc_initialization,1,2022-04-19,2022-10-14,2022-04-19,2022-10-14,125,15.0726,0.150726,-15.0726,0.116676,1.000465,0.849669
7,wf_2022,reward_ppo_selected,1,2022-04-19,2022-10-14,2022-04-19,2022-10-14,125,15.0726,0.150726,-15.0726,0.116676,1.000465,0.849669
8,wf_2023_2024,bc_initialization,1,2024-07-10,2024-12-19,2024-07-10,2024-12-19,115,13.9594,0.139594,-13.9594,0.162417,1.263423,1.087056
9,wf_2023_2024,bc_initialization,2,2024-06-18,2024-06-24,2024-06-18,2024-06-24,4,9.5583,0.095583,-9.5583,0.064421,1.209409,1.093810


,fold,interval_type,feature_group,feature_count,mean_abs_delta_z,max_abs_delta_z,mean_delta_z,mean_position_delta,mean_return_delta,top_features
0,wf_2020,worst_rolling_ppo_vs_bc_loss,other,2,1.071620,1.794457,1.071620,0.000422,-14.035033,"anchor_gap_180,anchor_gap_180,anchor_gap_180,a..."
1,wf_2020,worst_rolling_ppo_vs_bc_loss,volatility_or_stress,5,0.832796,1.664624,-0.496524,0.000422,-14.035033,"vix_percentile_252,vix_percentile_252,vix_perc..."
2,wf_2020,worst_rolling_ppo_vs_bc_loss,relative_strength_spread,1,0.631738,0.652290,0.631738,0.000422,-14.035033,"relative_strength_21,relative_strength_21,rela..."
3,wf_2020,worst_rolling_ppo_vs_bc_loss,trend_or_momentum,4,0.574177,0.888741,0.574177,0.000422,-14.035033,"trend_gap_21,trend_gap_21,trend_gap_21,mom_21"
4,wf_2022,worst_rolling_ppo_vs_bc_loss,other,2,2.580658,3.416262,2.580658,0.000000,0.000000,"anchor_gap_180,anchor_gap_180,anchor_gap_180,a..."
5,wf_2022,worst_rolling_ppo_vs_bc_loss,trend_or_momentum,4,1.013479,2.104659,0.551694,0.000000,0.000000,"mom_63,mom_63,mom_63,trend_gap_63"
6,wf_2022,worst_rolling_ppo_vs_bc_loss,volatility_or_stress,5,0.799857,1.388353,-0.093210,0.000000,0.000000,"realized_vol_21,realized_vol_21,drawdown_63,re..."
7,wf_2022,worst_rolling_ppo_vs_bc_loss,relative_strength_spread,1,0.386713,0.477618,-0.386713,0.000000,0.000000,"relative_strength_21,relative_strength_21,rela..."
8,wf_2023_2024,worst_rolling_ppo_vs_bc_loss,trend_or_momentum,4,0.922678,1.450151,0.922678,0.000000,0.000000,"trend_gap_63,mom_63,trend_gap_63,trend_gap_63"
9,wf_2023_2024,worst_rolling_ppo_vs_bc_loss,volatility_or_stress,5,0.793834,1.175966,-0.402994,0.000000,0.000000,"vix_percentile_252,vix_percentile_252,vix_perc..."


In [7]:
state_rows: list[dict[str, Any]] = []
available_key_features = [feature for feature in KEY_FEATURES if feature in daily_delta.columns]
for fold in WALK_FORWARD_WINDOWS:
    sub = daily_delta[daily_delta["fold"].eq(fold)].copy()
    if sub.empty:
        continue
    for feature in available_key_features:
        values = pd.to_numeric(sub[feature], errors="coerce")
        if values.dropna().nunique() < 3:
            continue
        try:
            bins = pd.qcut(values.rank(method="first"), q=4, labels=["q1_low", "q2", "q3", "q4_high"])
        except ValueError:
            continue
        sub_feature = sub.assign(feature_bin=bins)
        for feature_bin, group in sub_feature.groupby("feature_bin", observed=True):
            if group.empty:
                continue
            state_rows.append(
                {
                    "fold": fold,
                    "feature": feature,
                    "feature_bin": str(feature_bin),
                    "rows": int(len(group)),
                    "feature_min": float(pd.to_numeric(group[feature], errors="coerce").min()),
                    "feature_max": float(pd.to_numeric(group[feature], errors="coerce").max()),
                    "avg_bc_pos": float(group["position_bc"].mean()),
                    "avg_ppo_pos": float(group["position_ppo"].mean()),
                    "avg_position_delta": float(group["position_delta"].mean()),
                    "action_changed_rate": float(group["action_changed"].mean()),
                    "bc_return_pct": path_return_pct(group["bc_nav"]),
                    "ppo_return_pct": path_return_pct(group["ppo_nav"]),
                    "ppo_minus_bc_return_pp": path_return_pct(group["ppo_nav"]) - path_return_pct(group["bc_nav"]),
                    "buyhold_return_pct": path_return_pct(group["buyhold_nav_rebased"]),
                }
            )
state_position_bins = pd.DataFrame(state_rows)
if not state_position_bins.empty:
    state_position_bins = state_position_bins.sort_values(["fold", "ppo_minus_bc_return_pp", "feature", "feature_bin"]).reset_index(drop=True)
state_position_bins.to_csv(CARD / "state_position_bins.csv", index=False)

display(state_position_bins.head(30))


,fold,feature,feature_bin,rows,feature_min,feature_max,avg_bc_pos,avg_ppo_pos,avg_position_delta,action_changed_rate,bc_return_pct,ppo_return_pct,ppo_minus_bc_return_pp,buyhold_return_pct
0,wf_2020,drawdown_63,q4_high,63,-0.017982,0.000000,0.436865,0.449612,0.012747,0.111111,90.738070,47.917384,-42.820686,64.200565
1,wf_2020,mom_21,q4_high,63,0.276530,0.860667,0.427929,0.428120,0.000191,0.174603,82.468610,42.904661,-39.563949,56.404983
2,wf_2020,relative_strength_21,q4_high,63,1.838910,5.732684,0.427330,0.427536,0.000206,0.174603,80.712434,42.112961,-38.599474,54.846551
3,wf_2020,anchor_gap_180,q4_high,63,0.544213,1.143732,0.377194,0.377488,0.000295,0.238095,80.375210,42.832748,-37.542462,58.093571
4,wf_2020,anchor_slope_180,q4_high,63,0.071491,0.195947,0.296876,0.275602,-0.021274,0.111111,80.375210,42.832748,-37.542462,58.093571
5,wf_2020,drawdown_63,q3,63,-0.084743,-0.019460,0.417632,0.406001,-0.011632,0.126984,80.375210,42.832748,-37.542462,67.120466
6,wf_2020,mom_21,q2,63,0.031128,0.178793,0.381020,0.352106,-0.028915,0.079365,80.375210,42.832748,-37.542462,57.135566
7,wf_2020,mom_63,q4_high,63,0.645145,1.360047,0.506929,0.516734,0.009805,0.095238,80.375210,42.832748,-37.542462,58.093571
8,wf_2020,realized_vol_21,q1_low,63,0.439632,0.732505,0.404169,0.388006,-0.016162,0.095238,80.375210,42.832748,-37.542462,58.093571
9,wf_2020,regime_risk,q1_low,63,0.143929,0.361587,0.407047,0.414302,0.007255,0.126984,80.375210,42.832748,-37.542462,58.093571


In [8]:
wf2020_rolling = rolling_loss_windows[rolling_loss_windows["fold"].eq("wf_2020")].copy()
wf2020_worst = wf2020_rolling.iloc[0].to_dict() if not wf2020_rolling.empty else {}
wf2020_position = position_summary[(position_summary["fold"].eq("wf_2020"))].copy()
wf2020_bc = wf2020_position[wf2020_position["candidate"].eq("bc_initialization")].iloc[0].to_dict()
wf2020_ppo = wf2020_position[wf2020_position["candidate"].eq("reward_ppo_selected")].iloc[0].to_dict()

ppo_gate_count = int(position_summary[
    (position_summary["candidate"].eq("reward_ppo_selected"))
    & (position_summary["relative_return_pp"] >= TARGET_RELATIVE_RETURN)
    & (position_summary["max_dd_pct"] <= TARGET_MAX_DD)
].shape[0])
bc_gate_count = int(position_summary[
    (position_summary["candidate"].eq("bc_initialization"))
    & (position_summary["relative_return_pp"] >= TARGET_RELATIVE_RETURN)
    & (position_summary["max_dd_pct"] <= TARGET_MAX_DD)
].shape[0])

wf2020_rel_delta = float(wf2020_ppo["relative_return_pp"] - wf2020_bc["relative_return_pp"])
wf2020_pos_delta = float(wf2020_ppo["avg_pos"] - wf2020_bc["avg_pos"])
wf2020_action_change_rate = float(daily_delta[daily_delta["fold"].eq("wf_2020")]["action_changed"].mean())

if wf2020_rel_delta < -10 and wf2020_pos_delta <= 0:
    decision = "reward_ppo_wf2020_underparticipation_action_drift_no_promotion"
    next_mechanism = "state_conditioned_trend_participation_checkpoint_or_auxiliary_action_head_before_reward_ppo"
elif wf2020_rel_delta < -10:
    decision = "reward_ppo_wf2020_wrong_state_exposure_no_promotion"
    next_mechanism = "drawdown_interval_state_feature_or_checkpoint_selection_before_reward_ppo"
else:
    decision = "reward_ppo_action_diagnostic_no_clear_wf2020_cause_no_promotion"
    next_mechanism = "aggregate_poor_windows_and_feature_groups_before_new_training"

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "source_experiment": "release16_guarded_reward_short_ppo_screen_v1",
    "source_decision": source_summary.get("decision"),
    "comparison_pair": {
        "latest": "reward_ppo_selected",
        "previous": "bc_initialization",
        "baseline_family": "soxlbacktest_policy_reimplementation_v1 / release16 bull-reentry BC warm-start",
    },
    "gate_counts": {
        "reward_ppo_selected_20dd": ppo_gate_count,
        "bc_initialization_20dd": bc_gate_count,
        "window_count": len(WALK_FORWARD_WINDOWS),
    },
    "wf2020_diagnostic": {
        "bc_relative_return_pp": float(wf2020_bc["relative_return_pp"]),
        "ppo_relative_return_pp": float(wf2020_ppo["relative_return_pp"]),
        "relative_delta_pp": wf2020_rel_delta,
        "bc_max_dd_pct": float(wf2020_bc["max_dd_pct"]),
        "ppo_max_dd_pct": float(wf2020_ppo["max_dd_pct"]),
        "bc_avg_pos": float(wf2020_bc["avg_pos"]),
        "ppo_avg_pos": float(wf2020_ppo["avg_pos"]),
        "avg_position_delta": wf2020_pos_delta,
        "action_change_rate": wf2020_action_change_rate,
        "worst_rolling_loss_window": wf2020_worst,
    },
    "top_feature_groups": feature_groups.head(8).to_dict("records") if not feature_groups.empty else [],
    "output_paths": {
        "notebook": rel(CARD / f"{EXPERIMENT}.ipynb"),
        "position_action_summary": rel(CARD / "position_action_summary.csv"),
        "daily_position_deltas": rel(CARD / "daily_position_deltas.csv"),
        "rolling_loss_windows": rel(CARD / "rolling_loss_windows.csv"),
        "drawdown_intervals": rel(CARD / "drawdown_intervals.csv"),
        "feature_deltas": rel(CARD / "feature_deltas.csv"),
        "feature_group_summary": rel(CARD / "feature_group_summary.csv"),
        "state_position_bins": rel(CARD / "state_position_bins.csv"),
        "selection": rel(CARD / "selection.json"),
        "summary": rel(CARD / "summary.json"),
        "conclusion": rel(CARD / "conclusion.md"),
    },
    "conclusion": [],
}
summary["conclusion"] = [
    f"Reward PPO remains non-promotable: {ppo_gate_count}/{len(WALK_FORWARD_WINDOWS)} primary 20% gates, versus BC {bc_gate_count}/{len(WALK_FORWARD_WINDOWS)}.",
    f"wf_2020 is the critical regression: relative return delta is {wf2020_rel_delta:.2f}pp while MaxDD is roughly unchanged ({wf2020_bc['max_dd_pct']:.2f}% -> {wf2020_ppo['max_dd_pct']:.2f}%).",
    f"wf_2020 average position changed by {wf2020_pos_delta:.3f}; action_changed_rate={wf2020_action_change_rate:.1%}.",
    f"Worst 21-day PPO-vs-BC window: {wf2020_worst.get('start', '')} to {wf2020_worst.get('end', '')}, delta={float(wf2020_worst.get('ppo_minus_bc_return_pp', 0.0)):.2f}pp, avg position delta={float(wf2020_worst.get('avg_position_delta', 0.0)):.3f}.",
    "No actor is promoted or retained; the next step should change state/action objective before another reward PPO run.",
]
selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "wf2020_relative_delta_pp": wf2020_rel_delta,
    "wf2020_avg_position_delta": wf2020_pos_delta,
    "wf2020_action_change_rate": wf2020_action_change_rate,
    "reward_ppo_gate_count_20dd": ppo_gate_count,
    "bc_gate_count_20dd": bc_gate_count,
    "source_decision": source_summary.get("decision"),
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)

conclusion_md = chr(10).join(
    [
        "# Conclusion",
        "",
        f"- decision: `{decision}`",
        f"- reward_ppo_gate_count_20dd: `{ppo_gate_count}/{len(WALK_FORWARD_WINDOWS)}`",
        f"- bc_gate_count_20dd: `{bc_gate_count}/{len(WALK_FORWARD_WINDOWS)}`",
        f"- promotion_allowed: `False`",
        f"- actor_retention_allowed: `False`",
        "",
        "## Readout",
        "",
        *[f"- {line}" for line in summary["conclusion"]],
        "",
        "## Next Action",
        "",
        f"- `{next_mechanism}`",
    ]
) + chr(10)
(CARD / "conclusion.md").write_text(conclusion_md, encoding="utf-8")
display(Markdown(conclusion_md))
summary


# Conclusion

- decision: `reward_ppo_wf2020_underparticipation_action_drift_no_promotion`
- reward_ppo_gate_count_20dd: `1/3`
- bc_gate_count_20dd: `1/3`
- promotion_allowed: `False`
- actor_retention_allowed: `False`

## Readout

- Reward PPO remains non-promotable: 1/3 primary 20% gates, versus BC 1/3.
- wf_2020 is the critical regression: relative return delta is -37.54pp while MaxDD is roughly unchanged (21.11% -> 21.11%).
- wf_2020 average position changed by -0.027; action_changed_rate=10.7%.
- Worst 21-day PPO-vs-BC window: 2020-11-05 to 2020-12-04, delta=-14.36pp, avg position delta=0.000.
- No actor is promoted or retained; the next step should change state/action objective before another reward PPO run.

## Next Action

- `state_conditioned_trend_participation_checkpoint_or_auxiliary_action_head_before_reward_ppo`


{'experiment': 'release16_guarded_reward_action_feature_diagnostic_v1',
 'created_at': '2026-06-14T21:13:32.545356+00:00',
 'decision': 'reward_ppo_wf2020_underparticipation_action_drift_no_promotion',
 'next_mechanism': 'state_conditioned_trend_participation_checkpoint_or_auxiliary_action_head_before_reward_ppo',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'source_experiment': 'release16_guarded_reward_short_ppo_screen_v1',
 'source_decision': 'release16_guarded_reward_short_ppo_drift_failed_no_promotion',
 'comparison_pair': {'latest': 'reward_ppo_selected',
  'previous': 'bc_initialization',
  'baseline_family': 'soxlbacktest_policy_reimplementation_v1 / release16 bull-reentry BC warm-start'},
 'gate_counts': {'reward_ppo_selected_20dd': 1,
  'bc_initialization_20dd': 1,
  'window_count': 3},
 'wf2020_diagnostic': {'bc_relative_return_pp': 22.281638935298567,
  'ppo_relative_return_pp': -15.260822889766978,
  'relative_delta_pp': -37.542461825065544,
  'bc_max_d